# Notebook 2: One neuron (pass/fail with logistic regression)

Step 2 of the ladder. Notebook 1 gave you the machine: **parameters, a loss, and a search** (gradient descent). That notebook fit a line to *numbers*. Here the answer is a **category** (pass or fail, left or right), and we meet the unit every neural network is built from: **one neuron**.

You need Notebook 1's one-line update `w = w - lr * slope`. Nothing else is assumed. NumPy and matplotlib only, runs in seconds.

### What you'll be able to say at the end
- A neuron is **a weighted sum, squashed into a probability**.
- Why classification needs a *different loss* than fitting a line, and the neat fact that the slopes come out looking almost identical.
- What one neuron **cannot** do (and the cheap fix, and the real fix we build in Notebook 4).
- Why the CartPole policy in REINFORCE **is** a neuron, and why its update is the same slope you compute here.

Same markers as before: `WHAT / WHY / SAME AS / BECOMES / WATCH` comments, **PREDICT** cells, a knob ledger, and a sandbox at the end.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

# WHAT: plot styling only (same quiet look as Notebook 1). Nothing here affects the math.
BLUE, ORANGE, AQUA, YELLOW = "#2a78d6", "#eb6834", "#1baf7a", "#eda100"
INK, MUTED, GRIDC = "#0b0b0b", "#52514e", "#e1e0d9"
BLUE_RAMP = ["#86b6ef", "#3987e5", "#1c5cab", "#0d366b"]       # light -> dark = early -> late
plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb", "savefig.facecolor": "#fcfcfb",
    "axes.edgecolor": "#c3c2b7", "axes.labelcolor": MUTED, "xtick.color": MUTED, "ytick.color": MUTED,
    "text.color": INK, "axes.titlecolor": INK, "axes.titlesize": 11, "axes.labelsize": 10, "font.size": 10,
    "axes.grid": True, "grid.color": GRIDC, "grid.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False,
    "lines.linewidth": 2.0, "legend.frameon": False, "legend.fontsize": 9, "figure.dpi": 110,
})

def tag(ax, x, y, text, color, dx=4, dy=0, ha="left"):
    # WHAT: write a small label next to a line instead of making you decode a legend.
    ax.annotate(text, (x, y), xytext=(dx, dy), textcoords="offset points", color=color, fontsize=9, va="center", ha=ha)

def slope_by_nudging(f, w, h=1e-5):
    # SAME AS: Notebook 1. Nudge and measure: works on ANY function, so it is our check on every formula below.
    return (f(w + h) - f(w - h)) / (2 * h)

print("ready")

ready


# Part 1: Pass/fail data, and why a plain line fails

> **You are here:** a new kind of answer. Not "how much" but "which one".
> **The setup:** a sensor reading `x` on each part, and whether the part passed inspection (`y = 1`) or failed (`y = 0`). Low readings tend to fail, high readings tend to pass, with a fuzzy overlap in the middle (real measurements are never clean).

Notebook 1's tool was a line `y_hat = m*x + b` fit by squared error. Try it on pass/fail and see what goes wrong.

In [2]:
rng = np.random.default_rng(1)
n = 60
x1d = rng.uniform(-4, 4, n)                              # WHAT: 60 sensor readings between -4 and 4
p_true = 1 / (1 + np.exp(-(1.5 * x1d - 0.5)))            # WHAT: the hidden truth: chance of passing climbs smoothly with x
y1d = (rng.random(n) < p_true).astype(float)             # WHAT: each part passes (1) or fails (0) with that chance. Fuzzy, like real life.

m, b = np.polyfit(x1d, y1d, 1)                           # WHAT: Notebook 1's tool: best straight line through the 0/1 labels
line_pred = m * x1d + b
print(f"passes: {int(y1d.sum())} of {n}")
print(f"the straight-line 'prediction' ranges from {line_pred.min():.2f} to {line_pred.max():.2f}")
print(f"parts where the line predicts a 'probability' outside 0..1: {int(np.sum((line_pred < 0) | (line_pred > 1)))} of {n}")

passes: 29 of 60
the straight-line 'prediction' ranges from -0.25 to 1.14
parts where the line predicts a 'probability' outside 0..1: 17 of 60


In [3]:
fig, ax = plt.subplots(figsize=(7.2, 3.9))
jitter = (np.random.default_rng(5).random(n) - 0.5) * 0.06     # WHAT: nudge the dots up/down a hair so they do not hide each other
ax.scatter(x1d, y1d + jitter, s=20, color=MUTED, zorder=3)
xs = np.linspace(-4, 4, 100)
ax.plot(xs, m * xs + b, color=ORANGE)
tag(ax, 4, m * 4 + b, "straight line", ORANGE, dx=-4, dy=10, ha="right")
ax.axhline(0, color="#c3c2b7", linewidth=1); ax.axhline(1, color="#c3c2b7", linewidth=1)
ax.set_ylim(-0.4, 1.5); ax.set_xlabel("sensor reading x"); ax.set_ylabel("passed? (1 = yes)")
ax.set_title("A straight line escapes 0..1: it is not a probability")
plt.show()

<Figure size 792x429 with 1 Axes>

**What to notice.** The line works as a rough trend, but it predicts about -0.25 and 1.14 at the ends (the printout above). A probability can't be below 0 or above 1. A line also keeps climbing forever, while real pass/fail flattens: a reading far above the threshold is *certainly* a pass, not "more than certainly".

We want a curve that goes **flat at 0 on the left, flat at 1 on the right, and smoothly rises in between.** That curve has a name.

# Part 2: The squash (the sigmoid)

> **You are here:** you need something that turns *any number* into a probability.

The **sigmoid** takes any number `z` (from minus infinity to plus infinity) and squashes it into the range 0 to 1: `sigmoid(z) = 1 / (1 + e^-z)`. Look at what it does to a few values:

In [4]:
def sigmoid(z):
    # WHAT: squash any number into 0..1.  Big negative -> near 0.  Zero -> exactly 0.5.  Big positive -> near 1.
    # WHY:  we want an output we can read as "probability of passing".
    # WATCH: np.clip keeps e^-z from overflowing to infinity for very negative z.
    z = np.clip(z, -500, 500)
    return 1 / (1 + np.exp(-z))

print("    z     | sigmoid(z)")
for z in [-6, -2, -1, 0, 1, 2, 6]:
    print(f"  {z:+3d}     | {sigmoid(z):.4f}")
print()
print("sigmoid(z) + sigmoid(-z) =", sigmoid(2.0) + sigmoid(-2.0), "  (it is symmetric: the 'fail' chance mirrors the 'pass' chance)")

    z     | sigmoid(z)
   -6     | 0.0025
   -2     | 0.1192
   -1     | 0.2689
   +0     | 0.5000
   +1     | 0.7311
   +2     | 0.8808
   +6     | 0.9975

sigmoid(z) + sigmoid(-z) = 0.9999999999999999   (it is symmetric: the 'fail' chance mirrors the 'pass' chance)


**What to notice.** `z = 0` gives exactly 0.5 ("can't tell"). `z = +6` gives 0.9975 and `z = -6` gives 0.0025. The curve is steepest in the middle and flat at the ends, so far from the decision point the output barely changes. Remember that flatness. It comes back in Part 4 and causes real trouble.

**A neuron is just this:**
1. a weighted sum `z = w*x + b` (the same line as before, with `w` as the slope and `b` as the intercept),
2. squashed by the sigmoid: `p = sigmoid(w*x + b)`.

The output `p` is the neuron's **probability of passing.** `w` says how sharply the reading separates pass from fail, and `-b/w` is the reading where it's a coin flip. By hand, with `w = 1.5` and `b = -0.5`:

In [5]:
w, b = 1.5, -0.5                       # WHAT: the neuron's two parameters (the same two knobs-that-get-learned as the line's slope/intercept)
for reading in (-2.0, 0.0, 1.0, 3.0):
    z = w * reading + b                # WHAT: step 1: the weighted sum
    p = sigmoid(z)                     # WHAT: step 2: the squash
    print(f"reading {reading:+.1f}:  z = {w}*{reading:+.1f} + ({b}) = {z:+.2f}   ->  p(pass) = {p:.3f}")
print(f"coin-flip reading (z = 0): x = -b/w = {-b / w:.3f}")

reading -2.0:  z = 1.5*-2.0 + (-0.5) = -3.50   ->  p(pass) = 0.029
reading +0.0:  z = 1.5*+0.0 + (-0.5) = -0.50   ->  p(pass) = 0.378
reading +1.0:  z = 1.5*+1.0 + (-0.5) = +1.00   ->  p(pass) = 0.731
reading +3.0:  z = 1.5*+3.0 + (-0.5) = +4.00   ->  p(pass) = 0.982
coin-flip reading (z = 0): x = -b/w = 0.333


# Part 3: Squared error is the wrong gauge (a trap worth seeing once)

> **You are here:** the neuron makes a probability. Now it needs a loss so gradient descent can train it.

The obvious loss from Notebook 1 is squared error `(p - y)^2`. It works in principle, but it hides a nasty trap, and the trap is a good lesson about **why the loss matters to the search**.

Take one part with reading `x = 1` that really passed (`y = 1`), and a neuron that starts out **confidently wrong**: `w = -6`, so `p = sigmoid(-6) = 0.0025`. Measure the slope of the loss (by nudging) for squared error and for the alternative, **cross-entropy**: `-log(p)` when the part passed, `-log(1 - p)` when it failed. Cross-entropy punishes confident wrongness *enormously*: `-log(0.0025)` is about 6, and it heads to infinity as `p` heads to 0.

In [6]:
x_one, y_one = 1.0, 1.0                               # one reading, and the part really passed

def loss_mse_one(w):
    return (sigmoid(w * x_one) - y_one) ** 2          # WHAT: squared-error loss for this one part

def loss_ce_one(w):
    p = sigmoid(w * x_one)
    return -(y_one * np.log(p) + (1 - y_one) * np.log(1 - p))   # WHAT: cross-entropy: -log of the probability given to the truth
    # WHY:  if it was a pass (y=1) this is -log(p); if a fail (y=0) it is -log(1-p). Confident and wrong = huge loss.

print("  w   |  p(pass)  | slope of MSE loss | slope of cross-entropy loss")
for w_ in (-6, -4, -2, 0, 2):
    print(f" {w_:+3d}  |  {sigmoid(w_ * x_one):.4f}   | {slope_by_nudging(loss_mse_one, w_):+17.5f} | {slope_by_nudging(loss_ce_one, w_):+.5f}")

  w   |  p(pass)  | slope of MSE loss | slope of cross-entropy loss
  -6  |  0.0025   |          -0.00492 | -0.99753
  -4  |  0.0180   |          -0.03469 | -0.98201
  -2  |  0.1192   |          -0.18496 | -0.88080
  +0  |  0.5000   |          -0.25000 | -0.50000
  +2  |  0.8808   |          -0.02503 | -0.11920


**What to notice.** At `w = -6` (confidently wrong) the squared-error slope is essentially **zero** (-0.005), while the cross-entropy slope is **-1.0**. A near-zero slope means gradient descent's step `lr * slope` is nearly nothing, so a squared-error neuron that starts out confidently wrong barely moves. This is the flat end of the sigmoid from Part 2 biting: the squash flattens the gauge exactly where the neuron is most wrong.

Prove it by running both from `w = -6`:

In [7]:
def train_one(loss_fn, w0=-6.0, lr=0.5, steps=200):
    w = w0
    for _ in range(steps):
        w = w - lr * slope_by_nudging(loss_fn, w)       # SAME AS: Notebook 1's one-line update
    return w

w_mse = train_one(loss_mse_one)
w_ce = train_one(loss_ce_one)
print(f"after 200 steps from w = -6 with lr = 0.5:")
print(f"   squared error  : w = {w_mse:+.3f}  ->  p(pass) = {sigmoid(w_mse):.4f}   (barely started to fix the mistake)")
print(f"   cross-entropy  : w = {w_ce:+.3f}  ->  p(pass) = {sigmoid(w_ce):.4f}   (fixed)")

after 200 steps from w = -6 with lr = 0.5:
   squared error  : w = -5.327  ->  p(pass) = 0.0048   (barely started to fix the mistake)
   cross-entropy  : w = +4.507  ->  p(pass) = 0.9891   (fixed)


**What to notice.** Same data, same search, same learning rate. The only difference is the loss, and one of them trains while the other crawls. **The loss isn't only "what you want". It also decides whether the search has a usable slope to follow.** This is the objective family of knobs from Notebook 1, with a concrete reason behind the choice.

Cross-entropy is the standard loss for any "which category" or "which action" problem. You'll meet it again as the `-log π(action)` in REINFORCE.

# Part 4: The slope of cross-entropy (it looks familiar)

> **You are here:** you need the slope of the loss to train the neuron. The surprise is how clean it is.

For the neuron `p = sigmoid(w*x + b)` with cross-entropy loss averaged over the data, the slopes are:

`d(loss)/dw = mean( (p - y) * x )`   and   `d(loss)/db = mean( p - y )`

Compare with Notebook 1's line fit: `mean(2 * miss * x)` and `mean(2 * miss)`. **Same shape.** The "miss" is now `p - y` (predicted chance minus the truth), and the sigmoid and the log cancel each other's awkwardness. Never trust a formula you can check, so test it by nudging:

In [8]:
def neuron_loss(params, X, Y):
    # WHAT: average cross-entropy over all the data for the neuron p = sigmoid(w*x + b)
    w, b = params
    p = np.clip(sigmoid(w * X + b), 1e-12, 1 - 1e-12)      # WATCH: clip so log(0) can never happen
    return -np.mean(Y * np.log(p) + (1 - Y) * np.log(1 - p))

def neuron_grad(params, X, Y):
    w, b = params
    miss = sigmoid(w * X + b) - Y                           # WHAT: predicted chance minus the truth, for every part
    return np.array([np.mean(miss * X), np.mean(miss)])     # WHY:  the chain rule makes the sigmoid cancel; no extra terms

p_test = np.array([0.7, -0.2]); h = 1e-6
nudged = np.array([
    (neuron_loss(p_test + [h, 0], x1d, y1d) - neuron_loss(p_test - [h, 0], x1d, y1d)) / (2 * h),
    (neuron_loss(p_test + [0, h], x1d, y1d) - neuron_loss(p_test - [0, h], x1d, y1d)) / (2 * h),
])
print("slopes from my formulas :", np.round(neuron_grad(p_test, x1d, y1d), 6))
print("slopes by nudging       :", np.round(nudged, 6))

slopes from my formulas : [-0.26017   0.015151]
slopes by nudging       : [-0.26017   0.015151]


They agree. This is the cleanest fact in this notebook, so say it in words: **the push on each weight is (how wrong the probability was) times (the input that weight saw).** A neuron that said 0.9 for a part that failed gets pushed hard; one that said 0.51 for a pass gets nudged lightly. The same sentence describes every weight in every network we'll build.

# Part 5: Train it

> **You are here:** you have the model (sigmoid of a line), the loss (cross-entropy), and the slope. Notebook 1's loop does the rest.

**PREDICT first.** Starting from `w = 0, b = 0` (every part gets p = 0.5), will the curve at step 5 already be steep or still nearly flat? And roughly what slope `w` will it end up with (the truth used 1.5)? Type a guess.

In [9]:
# PREDICT: after 5 steps, steep or flat? Where will w end up near? (the data-generating truth was w=1.5, b=-0.5)
my_prediction = ""
print("your prediction:", my_prediction if my_prediction else "(blank: type one above first)")

your prediction: (blank: type one above first)


In [10]:
params = np.array([0.0, 0.0])
lr = 0.5                                             # WHAT: the learning rate, the search knob from Notebook 1
history, losses = [params.copy()], [neuron_loss(params, x1d, y1d)]
print(" step |    w     |    b     |  loss")
for step in range(1, 2001):
    params = params - lr * neuron_grad(params, x1d, y1d)       # SAME AS: Notebook 1, Part 3. The identical update line.
    history.append(params.copy()); losses.append(neuron_loss(params, x1d, y1d))
    if step in (1, 5, 20, 100, 500, 2000):
        print(f"  {step:4d} | {params[0]:8.4f} | {params[1]:8.4f} | {losses[-1]:.4f}")
w_fit, b_fit = params
acc = np.mean((sigmoid(w_fit * x1d + b_fit) > 0.5) == (y1d == 1))
print()
print(f"final: w = {w_fit:.3f}, b = {b_fit:.3f}   (truth used w = 1.5, b = -0.5; with only 60 noisy parts the fit wanders)")
print(f"coin-flip reading = -b/w = {-b_fit / w_fit:.3f}      accuracy at threshold 0.5: {acc:.0%}")

 step |    w     |    b     |  loss
     1 |   0.4652 |  -0.0083 | 0.3894
     5 |   0.9889 |  -0.0907 | 0.2545
    20 |   1.5487 |  -0.3428 | 0.2009
   100 |   2.2479 |  -0.8199 | 0.1788
   500 |   2.6068 |  -1.0016 | 0.1771


  2000 |   2.6154 |  -1.0053 | 0.1771

final: w = 2.615, b = -1.005   (truth used w = 1.5, b = -0.5; with only 60 noisy parts the fit wanders)
coin-flip reading = -b/w = 0.384      accuracy at threshold 0.5: 90%


In [11]:
history = np.array(history)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.9))
a1.scatter(x1d, y1d + jitter, s=18, color=MUTED, zorder=3)
xs = np.linspace(-4, 4, 200)
for step, color in zip((0, 5, 100, 2000), BLUE_RAMP):
    w_, b_ = history[step]
    a1.plot(xs, sigmoid(w_ * xs + b_), color=color)
    lab = {0: ("start", -8), 5: ("step 5", -9), 100: ("steps 100 and 2000", 9), 2000: (None, 0)}[step]
    if lab[0]: tag(a1, 4, sigmoid(w_ * 4 + b_), lab[0], color, dy=lab[1])
a1.set_xlim(-4.2, 7.2); a1.set_ylim(-0.1, 1.12); a1.set_xlabel("sensor reading x"); a1.set_ylabel("p(pass)")
a1.set_title("The squash curve steepens into place (darker = later)")
a2.plot(np.arange(1, len(losses)), losses[1:], color=BLUE)
a2.set_xscale("log"); a2.set_xlabel("step (log scale)"); a2.set_ylabel("cross-entropy loss")
a2.set_title("Loss falls, then flattens at the noise floor")
plt.tight_layout(); plt.show()

<Figure size 1210x429 with 2 Axes>

**What to notice.**
- The loss never reaches zero. The data is fuzzy, so even the best curve is uncertain in the overlap region. A model that drove the loss to zero on noisy data would be **memorizing the noise**, which is Notebook 3's subject.
- The fitted `w` and `b` aren't exactly the ones used to generate the data, and they shouldn't be: 60 noisy samples can't pin them down. That is **estimation noise**, and more data narrows it.
- Accuracy at a 0.5 threshold is one number summarizing a curve. The curve itself (the probabilities) carries more information, as Part 7 shows.

# Part 6: Two inputs, and the decision boundary

> **You are here:** real parts have many readings. A neuron takes any number of inputs: one weight per input.

`z = w1*x1 + w2*x2 + b`, then squash. The set of readings where `z = 0` (a coin flip) is a **straight line** in the `(x1, x2)` picture. That's the **decision boundary**. One side is "probably pass", the other "probably fail".

Data: two clusters. Passing parts centered near `(+1, +1)` and failing parts near `(-1, -1)`, overlapping a bit.

In [12]:
def make_blobs(n_each, seed):
    r = np.random.default_rng(seed)
    pas = r.normal([+1.0, +1.0], 1.0, (n_each, 2))      # WHAT: passing parts cluster around (+1, +1)
    fail = r.normal([-1.0, -1.0], 1.0, (n_each, 2))     # WHAT: failing parts cluster around (-1, -1)
    X = np.vstack([pas, fail]); Y = np.r_[np.ones(n_each), np.zeros(n_each)]
    idx = r.permutation(len(Y))
    return X[idx], Y[idx]

Xb, Yb = make_blobs(150, seed=3)
ntr = int(0.7 * len(Yb))
Xb_tr, Yb_tr, Xb_te, Yb_te = Xb[:ntr], Yb[:ntr], Xb[ntr:], Yb[ntr:]     # WHAT: hold back 30% the neuron NEVER trains on, to judge it fairly

def fit_logistic(X, Y, lr=0.5, steps=3000):
    # WHAT: the whole training loop for a neuron with any number of inputs. Same update as before, vectorized.
    # WHY:  we standardize the inputs first (Notebook 1, Part 5: centered and scaled inputs train fast and safely).
    mu, sd = X.mean(0), X.std(0) + 1e-9
    Z = (X - mu) / sd
    w, b = np.zeros(Z.shape[1]), 0.0
    for _ in range(steps):
        miss = sigmoid(Z @ w + b) - Y                    # WHAT: (p - y) for every part, as in Part 4
        w = w - lr * (Z.T @ miss) / len(Y)               # WHAT: each weight's push = mean( miss * the input it saw )
        b = b - lr * np.mean(miss)
    return lambda Xnew: sigmoid(((Xnew - mu) / sd) @ w + b), (w, b, mu, sd)

predict_b, info_b = fit_logistic(Xb_tr, Yb_tr)
p_tr, p_te = predict_b(Xb_tr), predict_b(Xb_te)
acc_tr = np.mean((p_tr > 0.5) == (Yb_tr == 1)); acc_te = np.mean((p_te > 0.5) == (Yb_te == 1))
print(f"training accuracy: {acc_tr:.1%}    held-out accuracy: {acc_te:.1%}")
print("(the two clusters overlap, so even the best possible straight boundary cannot get 100%)")

training accuracy: 91.9%    held-out accuracy: 93.3%
(the two clusters overlap, so even the best possible straight boundary cannot get 100%)


In [13]:
fig, ax = plt.subplots(figsize=(6.2, 5.0))
ax.scatter(*Xb_te[Yb_te == 1].T, s=26, color=BLUE, marker="o", label="passed")
ax.scatter(*Xb_te[Yb_te == 0].T, s=30, color=ORANGE, marker="^", label="failed")
g1, g2 = np.meshgrid(np.linspace(-4, 4, 200), np.linspace(-4, 4, 200))
gp = predict_b(np.column_stack([g1.ravel(), g2.ravel()])).reshape(g1.shape)
ax.contour(g1, g2, gp, levels=[0.5], colors=[INK], linewidths=1.6)
ax.contour(g1, g2, gp, levels=[0.1, 0.9], colors=["#898781"], linewidths=1.0, linestyles="--")
ax.set_xlabel("reading x1"); ax.set_ylabel("reading x2"); ax.set_xlim(-4, 4); ax.set_ylim(-4, 4)
ax.set_title("Held-out parts. Solid line: p = 0.5. Dashed: p = 0.1 and 0.9")
ax.legend(loc="lower right")
plt.show()

<Figure size 682x550 with 1 Axes>

**What to notice.** The solid line is the coin-flip boundary; the dashed lines are where the neuron is 90% sure either way. The three lines are **parallel**: a single neuron always makes a straight boundary, and only the *confidence* changes with distance from it. That straightness is the key limitation, and Part 8 breaks it on purpose.

# Part 7: The judging knob: where to draw the pass line

> **You are here:** the neuron outputs a probability. To act (ship the part or scrap it) you must pick a **threshold**.

0.5 isn't sacred. Suppose shipping a bad part (**false pass**) costs 10 times as much as scrapping a good one (**false fail**). Then you should demand more certainty before passing. This threshold isn't learned by gradient descent. It's a decision about costs, which makes it a **judging** knob. Sweep it on the held-out parts:

In [14]:
cost_false_pass, cost_false_fail = 10.0, 1.0          # WHAT: the business facts: a false pass is 10x worse than a false fail

def total_cost(p, Y, thr):
    say_pass = p > thr
    false_pass = np.sum(say_pass & (Y == 0))          # WHAT: we shipped a part that was bad
    false_fail = np.sum(~say_pass & (Y == 1))         # WHAT: we scrapped a part that was good
    return cost_false_pass * false_pass + cost_false_fail * false_fail, int(false_pass), int(false_fail)

print(" threshold | false passes | false fails | total cost")
thr_grid = np.round(np.arange(0.05, 0.96, 0.05), 2)
costs = {}
for thr in thr_grid:
    c, fp_, ff_ = total_cost(p_te, Yb_te, thr); costs[thr] = c
    if thr in (0.1, 0.3, 0.5, 0.7, 0.9):
        print(f"   {thr:.2f}    |     {fp_:3d}      |    {ff_:3d}      |   {c:6.1f}")
best_thr = min(costs, key=costs.get)
print()
print(f"cheapest threshold on this held-out set: {best_thr:.2f} (cost {costs[best_thr]:.1f})  vs  at 0.50: {costs[0.5]:.1f}")

 threshold | false passes | false fails | total cost
   0.10    |       8      |      0      |     80.0
   0.30    |       5      |      1      |     51.0
   0.50    |       3      |      3      |     33.0
   0.70    |       1      |      6      |     16.0
   0.90    |       0      |      9      |      9.0

cheapest threshold on this held-out set: 0.75 (cost 6.0)  vs  at 0.50: 33.0


**What to notice.** When false passes cost more, the cheapest threshold sits above 0.5 (you want the neuron very sure before it says "pass"). The trained weights didn't change at all, only the way we *act* on the probability. Two caveats worth keeping: this is only 90 held-out parts, so the exact best threshold is noisy; and picking the threshold on the same set you report cost on is mildly optimistic. In a real project you'd pick it on one set and score on another.

# Part 8: What one neuron cannot do

> **You are here:** a neuron draws a straight boundary. What if the truth isn't straight?

A classic example (XOR-style): a part passes when the two readings have the **same sign** (both positive or both negative), and fails when they disagree. The passes live in two opposite corners, so no single straight line can separate them.

**PREDICT:** what accuracy do you expect from one neuron here: 50%, 75%, or near 100%? Type a guess, then run.

In [15]:
# PREDICT: accuracy of a single neuron on the same-sign-passes problem?
my_prediction = ""
print("your prediction:", my_prediction if my_prediction else "(blank: type one above first)")

your prediction: (blank: type one above first)


In [16]:
def make_xor(n, seed):
    r = np.random.default_rng(seed)
    X = r.uniform(-2, 2, (n, 2))
    Y = (X[:, 0] * X[:, 1] > 0).astype(float)           # WHAT: pass when the two readings have the same sign
    return X, Y

Xx_tr, Yx_tr = make_xor(300, 11); Xx_te, Yx_te = make_xor(300, 12)

predict_x, _ = fit_logistic(Xx_tr, Yx_tr)
acc_xor = np.mean((predict_x(Xx_te) > 0.5) == (Yx_te == 1))
print(f"one neuron, raw readings          : held-out accuracy {acc_xor:.1%}   (a coin flip is 50%)")

def add_product(X):
    return np.column_stack([X, X[:, 0] * X[:, 1]])      # WHAT: hand the neuron a NEW input: x1 * x2 (positive exactly when signs agree)

predict_xp, _ = fit_logistic(add_product(Xx_tr), Yx_tr)
acc_xor_fixed = np.mean((predict_xp(add_product(Xx_te)) > 0.5) == (Yx_te == 1))
print(f"same neuron + the input x1 * x2   : held-out accuracy {acc_xor_fixed:.1%}")

one neuron, raw readings          : held-out accuracy 50.7%   (a coin flip is 50%)


same neuron + the input x1 * x2   : held-out accuracy 99.0%


In [17]:
fig, ax = plt.subplots(figsize=(5.8, 5.0))
ax.scatter(*Xx_te[Yx_te == 1].T, s=22, color=BLUE, marker="o", label="passed")
ax.scatter(*Xx_te[Yx_te == 0].T, s=26, color=ORANGE, marker="^", label="failed")
g1, g2 = np.meshgrid(np.linspace(-2, 2, 200), np.linspace(-2, 2, 200))
gp = predict_x(np.column_stack([g1.ravel(), g2.ravel()])).reshape(g1.shape)
ax.contour(g1, g2, gp, levels=[0.5], colors=[INK], linewidths=1.6)
ax.set_xlabel("reading x1"); ax.set_ylabel("reading x2"); ax.set_title("One neuron's straight boundary (black) cannot separate this")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=2)
plt.show()

<Figure size 638x550 with 1 Axes>

**What to notice.**
- One neuron is stuck near a coin flip. Whatever straight line it draws splits each pair of opposite corners.
- Giving it one extra input, the **product** `x1*x2`, fixes it. We *designed* a better input by hand. This is **feature engineering**, and for decades it was most of machine learning.
- The big idea of neural networks: instead of designing the inputs yourself, **stack neurons** so that the first layer *learns* useful inputs (like "x1 and x2 agree") for the next layer to use. That's Notebook 4.

# Part 9: The bridge to CartPole (your REINFORCE policy is a neuron)

> **You are here:** the reason this notebook sits before RL.

In CartPole the agent picks one of **two** actions (push left or right), given a state. A *policy* that gives "probability of pushing right" is `p = sigmoid(w · state + b)`, which is **exactly the neuron you just built.** (Many REINFORCE implementations use a 2-output softmax instead; that is the two-action version of the same idea, with the same shape of maths.)

REINFORCE's loss for the action it actually took is `-log(probability of that action)`, which is **cross-entropy with the action as the label**. So its slope is Part 4's again: `(p - a)` times the input. Check by nudging, with `a = 1` meaning "pushed right":

In [18]:
state = np.array([0.3, -0.8])                     # WHAT: a pretend 2-number state (like pole angle and angular speed)
wts = np.array([0.5, -0.4])                       # WHAT: the policy's weights (2 inputs, so 2 weights)

def neg_log_prob(wts, a):
    p_right = sigmoid(wts @ state)                # WHAT: the neuron's chance of pushing right
    return -(a * np.log(p_right) + (1 - a) * np.log(1 - p_right))   # WHAT: -log(probability of the action actually taken)

for a in (1, 0):
    p_right = sigmoid(wts @ state)
    formula = (p_right - a) * state               # WHAT: Part 4's slope: (p - label) * input, label = the action taken
    nudge = np.array([(neg_log_prob(wts + e * 1e-6, a) - neg_log_prob(wts - e * 1e-6, a)) / 2e-6 for e in np.eye(2)])
    print(f"action a = {a}:  formula {np.round(formula, 5)}   nudged {np.round(nudge, 5)}")

action a = 1:  formula [-0.11538  0.30769]   nudged [-0.11538  0.30769]
action a = 0:  formula [ 0.18462 -0.49231]   nudged [ 0.18462 -0.49231]


They match. Now the **only** difference between REINFORCE and ordinary classification is a **weight on each example**: instead of "make the labeled action more likely" for every row, REINFORCE says "make the action I took more likely **in proportion to how good the outcome was**" (the return, or in A2C the advantage). Good outcome: push toward that action. Bad outcome: push away from it.

Watch it learn with the smallest possible game. The state is whether the pole leans left (`s = -1`) or right (`s = +1`). Pushing toward the lean earns reward 1, otherwise 0. The policy is one neuron on `[s, 1]`. The update is `w += lr * (reward - baseline) * (a - p) * input`, where the baseline is the running average reward (that subtraction is the "advantage" idea from A2C, in its simplest form).

In [19]:
rl = np.random.default_rng(0)
wp = np.zeros(2)                                   # WHAT: policy weights for the inputs [s, 1]; start at 0 -> p(right) = 0.5 everywhere
baseline, lr_pg = 0.0, 0.2                         # WHAT: running average reward; policy learning rate (a search knob)
checkpoints = {}
rolling = []
print(" episode | p(correct push) | avg reward (last 200)")
for ep in range(1, 3001):
    s = rl.choice([-1.0, 1.0])                     # WHAT: the pole leans left or right
    feat = np.array([s, 1.0])
    p_right = sigmoid(wp @ feat)
    a = float(rl.random() < p_right)               # WHAT: the policy SAMPLES an action (this randomness is how it explores)
    reward = 1.0 if a == (1.0 if s > 0 else 0.0) else 0.0
    adv = reward - baseline                         # WHAT: better or worse than usual? (the advantage)
    wp = wp + lr_pg * adv * (a - p_right) * feat    # WHAT: REINFORCE update = Part 4's slope, weighted by the advantage
    baseline += 0.05 * (reward - baseline)          # WHAT: keep the running average reward up to date
    rolling.append(reward)
    if ep in (1, 50, 200, 500, 1000, 3000):
        p_correct = 0.5 * (sigmoid(wp @ [1.0, 1.0]) + (1 - sigmoid(wp @ [-1.0, 1.0])))
        checkpoints[ep] = p_correct
        print(f"  {ep:5d}  |     {p_correct:.3f}       |   {np.mean(rolling[-200:]):.3f}")

 episode | p(correct push) | avg reward (last 200)
      1  |     0.525       |   1.000
     50  |     0.866       |   0.780
    200  |     0.983       |   0.885
    500  |     0.995       |   0.985
   1000  |     0.996       |   0.995
   3000  |     0.999       |   1.000


**What to notice.** The policy starts at a coin flip (0.5) and climbs to nearly always pushing toward the lean, with no one ever telling it the rule. It only saw rewards. Everything in the update is a neuron's slope times an advantage. A deep-learning RL algorithm swaps the one neuron for a network and the little game for CartPole, and keeps this skeleton.

# Part 10: Sandbox (your ideas compete here)

> **You are here:** a neuron can only draw a straight line, but you can change *what you hand it*. How far can you get on three problems?

**The ruler.** A classifier is a function `clf(Xtrain, ytrain, Xtest)` that returns a probability of class 1 for each test row. It's scored on a **held-out set** by **log-loss** (cross-entropy; lower is better) and accuracy. Three problems: overlapping **blobs** (a line is right), **XOR** (opposite corners), and a **ring** (class 1 is inside a circle, so the boundary is curved).

Reference entries use the same `fit_logistic`, with extra hand-made inputs: the product `x1*x2`, and squares `x1^2`, `x2^2`.

In [20]:
def make_ring(n, seed):
    r = np.random.default_rng(seed)
    X = r.uniform(-2, 2, (n, 2))
    Y = (np.sum(X ** 2, axis=1) < 1.5).astype(float)    # WHAT: class 1 when the point is inside a circle of radius ~1.22
    return X, Y

blob_tr = make_blobs(150, 21); blob_te = make_blobs(150, 22)
problems2 = [("blobs", blob_tr, blob_te), ("xor", (Xx_tr, Yx_tr), (Xx_te, Yx_te)), ("ring", make_ring(300, 31), make_ring(300, 32))]

def score(clf):
    out = []
    for name, (Xtr, Ytr), (Xte, Yte) in problems2:
        p = np.clip(clf(Xtr, Ytr, Xte), 1e-9, 1 - 1e-9)
        logloss = -np.mean(Yte * np.log(p) + (1 - Yte) * np.log(1 - p))
        out.append((logloss, np.mean((p > 0.5) == (Yte == 1))))
    return out

def with_features(fn):
    # WHAT: wrap fit_logistic so it first turns each row (x1, x2) into whatever inputs fn builds.
    def clf(Xtr, Ytr, Xte):
        predict, _ = fit_logistic(fn(Xtr), Ytr)
        return predict(fn(Xte))
    return clf

def race2(entries):
    print(f"{'':36s}" + "".join(f"{nm:>20s}" for nm, _, _ in problems2))
    print(f"{'':36s}" + "".join(f"{'logloss / accuracy':>20s}" for _ in problems2))
    table = {}
    for label, clf in entries:
        res = score(clf); table[label] = res
        print(f"{label:36s}" + "".join(f"{f'{l:.3f} / {a:.0%}':>20s}" for l, a in res))
    return table

reference2 = [
    ("plain neuron: x1, x2", with_features(lambda X: X)),
    ("+ product x1*x2", with_features(lambda X: np.column_stack([X, X[:, 0] * X[:, 1]]))),
    ("+ squares x1^2, x2^2", with_features(lambda X: np.column_stack([X, X ** 2]))),
    ("+ product and squares", with_features(lambda X: np.column_stack([X, X[:, 0] * X[:, 1], X ** 2]))),
]
table2 = race2(reference2)

                                                   blobs                 xor                ring
                                      logloss / accuracy  logloss / accuracy  logloss / accuracy


plain neuron: x1, x2                         0.209 / 91%         0.692 / 51%         0.598 / 72%


+ product x1*x2                              0.209 / 91%         0.073 / 99%         0.596 / 72%


+ squares x1^2, x2^2                         0.211 / 90%         0.695 / 49%         0.049 / 99%


+ product and squares                        0.210 / 90%         0.083 / 97%         0.049 / 99%


**What to notice.** Each hand-made input fixes exactly the problem it matches, and adds nothing on the others (a useless input is almost harmless here; with small data an extra input can start to **overfit**, which is Notebook 3). The squares fix the ring, because "inside a circle" is `x1^2 + x2^2 < r`, a *straight* boundary in the squared inputs. The product fixes XOR. The all-in entry handles every problem, at the cost of guessing in advance which inputs to supply.

That guessing is the real limit: you had to *know* the shape of the problem. Neural networks learn the inputs for themselves.

### Your slot
Edit `my_classifier` and re-run. Ideas: other hand-made inputs (`abs(x1)`, `x1*x2` squared, distance from a point); a different learning rate or more steps inside `fit_logistic`; add a deliberately useless input (random noise) and see whether it hurts; or write a completely different classifier (nearest neighbours?) as long as it returns probabilities.

In [21]:
def my_classifier(Xtr, Ytr, Xte):
    # SANDBOX: replace the feature builder below with your own idea. It must return p(class 1) for every row of Xte.
    def my_features(X):
        return X                      # <- starts as the plain neuron. Try np.column_stack([X, ...your new inputs...])
    predict, _ = fit_logistic(my_features(Xtr), Ytr)
    return predict(my_features(Xte))

_ = race2(reference2 + [("MY CLASSIFIER", my_classifier)])

                                                   blobs                 xor                ring
                                      logloss / accuracy  logloss / accuracy  logloss / accuracy


plain neuron: x1, x2                         0.209 / 91%         0.692 / 51%         0.598 / 72%


+ product x1*x2                              0.209 / 91%         0.073 / 99%         0.596 / 72%


+ squares x1^2, x2^2                         0.211 / 90%         0.695 / 49%         0.049 / 99%


+ product and squares                        0.210 / 90%         0.083 / 97%         0.049 / 99%


MY CLASSIFIER                                0.209 / 91%         0.692 / 51%         0.598 / 72%


# Part 11: The knob ledger, self-check, and what's next

## Knob ledger (Notebooks 1 and 2)
Families: **objective** (what you optimize), **model** (what's inside), **search** (how you hunt), **experience** (what data/trials), **judging** (how you decide and score). The first three rows are from Notebook 1; the rest are new here.

In [22]:
rows = [
    ("learning rate (NB1)", "search", "too small: slow", "too big: blows up (limit set by curvature)"),
    ("batch size (NB1)", "search", "1: noisy, never settles", "all data: smooth but costly"),
    ("input scaling (NB1)", "experience", "unscaled: far slower", "standardized: fast and safe"),
    ("loss for categories", "objective",
     f"squared error from a confident mistake: p(pass) stays {sigmoid(w_mse):.3f} after 200 steps",
     f"cross-entropy: reaches p(pass) {sigmoid(w_ce):.3f}"),
    ("squash (activation)", "model", "none (a line): outputs leave 0..1", "sigmoid: a probability, but flat at the ends"),
    ("inputs you hand it", "model",
     f"raw x1, x2 on XOR: {acc_xor:.0%}", f"plus x1*x2: {acc_xor_fixed:.0%}"),
    ("decision threshold", "judging",
     f"0.5 with 10:1 costs: total {costs[0.5]:.0f}", f"best threshold {best_thr:.2f}: total {costs[best_thr]:.0f}"),
    ("policy learning rate (RL)", "search",
     "not tested here: try lr_pg = 0.001 in Part 9", "not tested here: try lr_pg = 5 in Part 9"),
]
print(f"{'knob':30s}{'family':12s}{'at one extreme':62s}{'at the other'}")
print("-" * 160)
for knob, fam, a, b in rows:
    print(f"{knob:30s}{fam:12s}{a:62s}{b}")

knob                          family      at one extreme                                                at the other
----------------------------------------------------------------------------------------------------------------------------------------------------------------
learning rate (NB1)           search      too small: slow                                               too big: blows up (limit set by curvature)
batch size (NB1)              search      1: noisy, never settles                                       all data: smooth but costly
input scaling (NB1)           experience  unscaled: far slower                                          standardized: fast and safe
loss for categories           objective   squared error from a confident mistake: p(pass) stays 0.005 after 200 stepscross-entropy: reaches p(pass) 0.989
squash (activation)           model       none (a line): outputs leave 0..1                             sigmoid: a probability, but flat at the ends
inputs y

## Self-check (answer in your own words before you read the answers)
| Question | Short answer |
|---|---|
| What is a neuron? | A weighted sum of the inputs plus a bias, passed through a squash (sigmoid) so the output is a probability. |
| Why not fit pass/fail with a plain line? | It predicts outside 0..1 and keeps climbing; the squash flattens at 0 and 1 like real pass/fail. |
| Why is squared error bad here? | The sigmoid is flat when the neuron is confidently wrong, so the loss has almost no slope there and the search barely moves. Cross-entropy keeps a strong slope. |
| What is the push on each weight? | (predicted chance minus the label) times the input that weight saw. |
| What's the decision boundary of one neuron? | A straight line (a flat plane with more inputs). Always. |
| How is REINFORCE related? | Its policy is a neuron; its loss is `-log(prob of the action taken)`; its slope is the one above, scaled by the return or advantage. |
| Why does the threshold change with costs? | It isn't learned. It's a business decision about which mistake is worse. |

## What's next
- You can now read the output layer of a CartPole policy: logits, squash, `-log π(a)`, weighted by an advantage.
- You know the limit of one neuron (a straight boundary) and the hand-made fix.

**Notebook 3: overfitting.** We stop trusting the training score. A model with enough hand-made inputs can memorize noise. You'll *see* training and held-out scores part ways, find where they split, and meet the knobs that control it (the model-size and regularization families). Then Notebook 4 stacks neurons so the network builds the inputs itself.